# Selecting interesting test dwarfs


The goal of this notebook is to identify a small subset of dwarf galaxies 
to validate catalogue analysis on.

we want to select dwarf galaxies with
- High and low S/N
- extreme and weak emission
- old and young
- metal poor and rich
- close and far
- massive and low mass
- of all morphologies

To acchieve this, we select dwarf galaxies with properties near manually placed points in different plots as to pick out interesting features in different dimensions. In general, we use STARLIGHT derived stellar properties 

In [ ]:
from astropy.table import Table
import numpy as np
import arya
import matplotlib.pyplot as plt

In [ ]:
arya.init("mnras")

In [ ]:
dwarf_cat_raw = Table.read("data/stellar_mass_emline_dwarfs.fits")

In [ ]:
def select_by_quantile(x, q=[0.001, 0.01, 0.16, 0.5, 0.84, 0.99, 0.999]):

    qs = np.quantile(x[np.isfinite(x)], q)
    idxs = []
    for qv in qs:
        delta = np.abs(x - qv)
        delta[~np.isfinite(delta)] = np.inf
        delta[exclude_galaxies] = np.inf
        idxs.append(int(np.argmin(delta)))

    return idxs

In [ ]:
exclude_galaxies = [
        252007,
    607361,563512, 
    499389,
    180253, 383630, 443404, 
    545778, 462877,
    390153, 216115, 
    507309, 560673,
    555341, 385692,  10289, 311617, 616376,
    499395,
    533845, 452739, 500580,
]
    

In [ ]:
dwarf_cat = dwarf_cat_raw

In [ ]:
x_bpt = np.log10(dwarf_cat["nii6583_flux"] / dwarf_cat["halpha_flux"])  - 0.15
y_bpt = np.log10(dwarf_cat["oiii5007_flux"] / dwarf_cat["hbeta_flux"]) - 1.4
r_bpt = x_bpt * y_bpt

In [ ]:
plt.hist(r_bpt[np.isfinite(r_bpt)], bins=np.linspace(-2, 2, 100))

In [ ]:
strat_vars = [
    dwarf_cat["logm_cigale"],
    dwarf_cat["age_cg"],
    dwarf_cat["mag_r"],
    dwarf_cat["z_cg"],
    r_bpt,
    dwarf_cat["mag_g"] - dwarf_cat["mag_r"],
    dwarf_cat["snr_med"],
    dwarf_cat["z"]
]
    

In [ ]:
strat_labels = [
    "stellar mass",
    "age",
    "r mag",
    "metallicity",
    "BPT",
    "g - r",
    "SNR",
    "z"
]

In [ ]:
dwarf_idxs = []

for x in strat_vars:
    dwarf_idxs = [*dwarf_idxs, *select_by_quantile(x)]

len(dwarf_idxs)

In [ ]:
dwarf_highlighted = dwarf_cat[dwarf_idxs]

In [ ]:
fig, ax = plt.subplots()

x = dwarf_cat["mag_g"] - dwarf_cat["mag_r"]
y = dwarf_cat["logm_cigale"]
plt.scatter(x, y, s=0.3, lw=0, alpha=0.01, color="k")


plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")
ax.set(
    xlim = (-0.5, 1.5),
    ylim = (6, 9.6),
    xlabel = "g - r [mag]",
    ylabel = r"$\log M_\star / M_\odot$ [CIGALE]",
)

plt.savefig("figures/dwarf_selected_Mstar_gr.png")

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log([NII] $\lambda$6583 / H$\alpha$)",
    ylabel = r"log([OIII] $\lambda$5007 / H$\beta$)",
    xlim = (-2, 1),
    ylim = (-1, 2),
)


x = np.log10(dwarf_cat["nii6583_flux"] / dwarf_cat["halpha_flux"])
y = np.log10(dwarf_cat["oiii5007_flux"] / dwarf_cat["hbeta_flux"])

plt.scatter(x, y, s=0.3, alpha=0.3, c="k", lw=0, clim=(0, 1))
plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")

# x_model = np.linspace(-2, 0.15, 1000)
# y_model = 0.61 / (x_model - 0.15) + 1.4 # Eq. 1 from Zou+2024, +0.1 dex modelling uncertainty shift in both directions
# plt.plot(x_model, y_model)

x_model = np.linspace(-2, -0.05, 1000)
y_model = 0.61 / (x_model - 0.05) + 1.3
plt.plot(x_model, y_model)
plt.savefig("figures/dwarf_selected_BPT.png")

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log($M_\star / M_\odot$)",
    ylabel = r"log(SFR / M$_\odot$ yr$^{-1}$)",
    xlim = (6, 9.6),
    ylim = (-3, 2),
)

x = dwarf_cat["logm_cigale"]
y = dwarf_cat["logsfr_cigale"] 

plt.scatter(x, y, s=0.1, color="black", alpha=0.03, lw=0)
plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")
plt.savefig("figures/dwarf_selected_SFR_vs_Mstar.png")

In [ ]:
target_ids = dwarf_cat["targetid"][dwarf_idxs]


In [ ]:
with open("analysis/selected_targetid.txt", "w") as f:
    f.write("targetid = [\n")

    for target_id in target_ids:
         f.write(f"    {target_id},\n")
    
    f.write("]")

In [ ]:
import tomllib

In [ ]:
with open("analysis/selected_targetid.txt", "rb") as f:
    h = tomllib.load(f)
    print(h)

# LSS image retrieval

In [ ]:
from astropy.nddata import CCDData

In [ ]:
from astropy.utils.data import download_file
from astropy.io import fits
from astropy.wcs import WCS


In [ ]:
def download_lss_image(ra, dec):
    link = f"https://www.legacysurvey.org/viewer/cutout.fits?ra={ra}&dec={dec}&layer=ls-dr11&pixscale=0.262&bands=giz"
    
    filename = download_file(link, cache=True, show_progress=False, timeout=120)
    hdu = fits.open(filename)[0]
    image = hdu.data
    hdr = hdu.header
    wcs = WCS(hdr)

    return CCDData(image, header=hdr, wcs=wcs, unit="nanomaggy")


In [ ]:
from astropy.nddata import block_reduce
def show_colour_image(image,
               figsize=(10, 10),
               dpi = None,
               clabel=None,
               clim=None,
               show_colorbar=True, show_ticks=True,
               fig=None, ax=None, input_ratio=None,
              subplot_kw=dict()):
    """
    Show an image in matplotlib with some basic astronomically-appropriate stretching.

    Parameters
    ----------
    image
        The image to show
    percl : number
        The percentile for the lower edge of the stretch (or both edges if ``percu`` is None)
    percu : number or None
        The percentile for the upper edge of the stretch (or None to use ``percl`` for both)
    figsize : 2-tuple
        The size of the matplotlib figure in inches
    """

    if dpi is None:
        dpi = image.shape[0] / figsize[0]

    if (fig is None and ax is not None) or (fig is not None and ax is None):
        raise ValueError('Must provide both "fig" and "ax" '
                         'if you provide one of them')
    elif fig is None and ax is None:
        if figsize is not None:
            # Rescale the fig size to match the image dimensions, roughly
            image_aspect_ratio = image.shape[0] / image.shape[1]
            figsize = (max(figsize) * image_aspect_ratio, max(figsize))

        fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi, subplot_kw=subplot_kw)


    # To preserve details we should *really* downsample correctly and
    # not rely on matplotlib to do it correctly for us (it won't).

    # So, calculate the size of the figure in pixels, block_reduce to
    # roughly that,and display the block reduced image.

    # Thanks, https://stackoverflow.com/questions/29702424/how-to-get-matplotlib-figure-size

    fig_size_pix = fig.get_size_inches() * fig.dpi

    ratio = (image.shape[0:2] // fig_size_pix).max()

    if ratio <= 1:
        ratio = 1
    else:
        print("downsampling image by", ratio)

    ratio = input_ratio or ratio

    reduced_data = block_reduce(image, ratio)

    reduced_data = (reduced_data - clim[0]) / (clim[1] - clim[0])

    reduced_data[reduced_data > 1] = 1
    reduced_data[reduced_data < 0] = 0
    
    # Divide by the square of the ratio to keep the flux the same in the
    # reduced image. We do *not* want to do this for images which are
    # masks, since their values should be zero or one.
    # reduced_data = reduced_data / ratio**2

    # Of course, now that we have downsampled, the axis limits are changed to
    # match the smaller image size. Setting the extent will do the trick to
    # change the axis display back to showing the actual extent of the image.
    extent = [0, image.shape[1], 0, image.shape[0]]


    im = ax.imshow(reduced_data, origin='lower',
                   extent=extent, aspect='equal')


    if not show_ticks:
        ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)


In [ ]:
ras = dwarf_cat[dwarf_idxs]["ra"]
decs = dwarf_cat[dwarf_idxs]["dec"]

In [ ]:
img = download_lss_image(ras[0], decs[0])

In [ ]:
def get_img_data(img):
    img_data = np.moveaxis(img.data, 0, -1)[:, :, ::-1]

    return img_data

In [ ]:
img_data = get_img_data(img)

In [ ]:
def color_clip_image(img, vmin=0, vmax=None, norm=np.arcsinh, scale=1):
    img_clipped = img.copy()
    I = np.sum(img, axis=2) / 3
    if vmax is None:
        h = np.quantile(I[np.isfinite(I)], 0.99)
    else:
        h = vmax
    l = vmin

    def f(I):
        x = (norm(I/scale) - norm(l/scale)) / (norm(h/scale) - norm(l/scale))
        return x
    for i in range(3):
        img_clipped[:, :, i] *= f(I) / I

    return img_clipped

In [ ]:
def ra_dec_axis():
    plt.xlabel(r"Right Ascension")
    plt.ylabel(r"Declination")

In [ ]:
zeropoint = [-0.00, 0.000, 0.00]

In [ ]:
len(dwarf_idxs)

In [ ]:
imgs = []

for i, idx in enumerate(dwarf_idxs):    
    print("loading image", i)
    img = download_lss_image(dwarf_cat["ra"][idx], dwarf_cat["dec"][idx])

    imgs.append(img)

In [ ]:
bad_imgs = [np.sum(get_img_data(img) == 0.) > 65_000 for img in imgs]

In [ ]:
exclude = np.array(dwarf_idxs)[bad_imgs]

In [ ]:
exclude

In [ ]:
n_cols = 7
n_rows = len(dwarf_idxs) // n_cols 

print(n_rows * n_cols)
fig, axs = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2*n_rows), sharex=True, sharey=True)

for i, idx in enumerate(dwarf_idxs):
    row = i // n_cols
    col = i % n_cols
    ax = axs[row, col]
    plt.sca(ax)
    
    img = imgs[i]
    img_data = get_img_data(img)
    
    show_colour_image(color_clip_image(img_data + zeropoint, scale=1e-2, vmax=1e1), 
                      clim=(0, 1), figsize=(3, 3), subplot_kw=dict(projection=img.wcs[0]), ax=ax, fig=fig)
    # plt.scatter(img_data.shape[0] / 2, img_data.shape[1] / 2)
    plt.title(idx)

    if col == 0:
        plt.ylabel(strat_labels[row])


plt.tight_layout()
plt.savefig("figures/dwarf_selected_cutouts.png")